
# Chia dữ liệu cân bằng – 49 file cho mỗi lớp, không oversample Normal

Mục tiêu:

\[
Normal = Inner = Outer = Ball = 49\ \text{file}
\]

Normal giữ nguyên 49 file, **không nhân đôi**.

Mỗi lớp lỗi cũng chỉ chọn tổng cộng 49 file từ 4 tải:

\[
0g,\ 6g,\ 20g,\ 35g
\]

Phân chia mỗi lớp:

\[
34\ Train + 7\ Validation + 8\ Test = 49
\]

Để các tải vẫn tương đối cân bằng, mỗi lớp lỗi được chia:

- `0g`: 9 Train + 2 Val + 2 Test = 13
- `6g`: 9 Train + 1 Val + 2 Test = 12
- `20g`: 8 Train + 2 Val + 2 Test = 12
- `35g`: 8 Train + 2 Val + 2 Test = 12

Tổng:

\[
13+12+12+12=49
\]

Như vậy số file giữa 4 lớp là cân bằng ngay từ file nguồn.


In [ ]:

# Ô 1 - Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")


In [ ]:

# Ô 2 - Import

from pathlib import Path
import random
import shutil

import pandas as pd


In [ ]:

# Ô 3 - Cấu hình

PROJECT_ROOT = Path(
    "/content/drive/MyDrive"
)

NORMAL_DIR = (
    PROJECT_ROOT
    / "normal"
)

FAULT_CLASS_DIRS = {
    "inner_race":
        PROJECT_ROOT
        / "inner_race",

    "outer_race":
        PROJECT_ROOT
        / "outer_race",

    "ball_fault":
        PROJECT_ROOT
        / "ball_fault",
}

LOADS = [
    "0g",
    "6g",
    "20g",
    "35g",
]

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "balanced_raw_split_49_each_class"
)

RANDOM_SEED = 42

# ============================================================
# NORMAL: 34 / 7 / 8 = 49
# ============================================================

NORMAL_TRAIN = 34
NORMAL_VAL = 7
NORMAL_TEST = 8

# ============================================================
# FAULT: tổng 49 file/lớp, phân bố gần đều 4 tải
# ============================================================

FAULT_TRAIN_PER_LOAD = {
    "0g": 9,
    "6g": 9,
    "20g": 8,
    "35g": 8,
}

FAULT_VAL_PER_LOAD = {
    "0g": 2,
    "6g": 1,
    "20g": 2,
    "35g": 2,
}

FAULT_TEST_PER_LOAD = {
    "0g": 2,
    "6g": 2,
    "20g": 2,
    "35g": 2,
}

COPY_FILES = True

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("COPY_FILES :", COPY_FILES)


In [ ]:

# Ô 4 - Hàm tiện ích

def speed_value(path):

    try:
        return float(
            path.stem
        )

    except ValueError:
        return float("inf")


def list_csv_files(folder):

    if not folder.exists():

        raise FileNotFoundError(
            f"Không tìm thấy thư mục:\n{folder}"
        )

    files = sorted(
        folder.glob("*.csv"),
        key=lambda path: (
            speed_value(path),
            path.name
        )
    )

    if not files:

        raise FileNotFoundError(
            f"Không có file CSV trong:\n{folder}"
        )

    return files


def copy_one_file(
    source_path,
    destination_dir
):

    destination_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    destination_path = (
        destination_dir
        / source_path.name
    )

    shutil.copy2(
        source_path,
        destination_path
    )

    return destination_path


In [ ]:

# Ô 5 - Chia Normal 34 Train / 7 Val / 8 Test

rng = random.Random(
    RANDOM_SEED
)

normal_files = list_csv_files(
    NORMAL_DIR
)

print(
    "Số file Normal tìm thấy:",
    len(normal_files)
)

needed_normal = (
    NORMAL_TRAIN
    + NORMAL_VAL
    + NORMAL_TEST
)

if len(normal_files) < needed_normal:

    raise ValueError(
        f"Normal chỉ có {len(normal_files)} file, "
        f"cần ít nhất {needed_normal}."
    )

normal_files = list(
    normal_files
)

rng.shuffle(
    normal_files
)

# Chỉ dùng đúng 49 file
normal_selected = normal_files[
    :needed_normal
]

normal_train_files = (
    normal_selected[
        :NORMAL_TRAIN
    ]
)

normal_val_files = (
    normal_selected[
        NORMAL_TRAIN:
        NORMAL_TRAIN + NORMAL_VAL
    ]
)

normal_test_files = (
    normal_selected[
        NORMAL_TRAIN + NORMAL_VAL:
    ]
)

print(
    "Normal Train      :",
    len(normal_train_files)
)

print(
    "Normal Validation :",
    len(normal_val_files)
)

print(
    "Normal Test       :",
    len(normal_test_files)
)


In [ ]:

# Ô 6 - Chia từng lớp lỗi thành đúng 49 file

fault_splits = {}

for class_index, (
    class_name,
    class_root
) in enumerate(
    FAULT_CLASS_DIRS.items()
):

    print(
        "\n" + "=" * 80
    )

    print(
        "LỚP:",
        class_name
    )

    class_split = {
        "train": [],
        "validation": [],
        "test": [],
    }

    for load_index, load_name in enumerate(
        LOADS
    ):

        folder = (
            class_root
            / load_name
        )

        files = list_csv_files(
            folder
        )

        local_rng = random.Random(
            RANDOM_SEED
            + class_index * 100
            + load_index
        )

        files = list(
            files
        )

        local_rng.shuffle(
            files
        )

        n_train = (
            FAULT_TRAIN_PER_LOAD[
                load_name
            ]
        )

        n_val = (
            FAULT_VAL_PER_LOAD[
                load_name
            ]
        )

        n_test = (
            FAULT_TEST_PER_LOAD[
                load_name
            ]
        )

        n_needed = (
            n_train
            + n_val
            + n_test
        )

        if len(files) < n_needed:

            raise ValueError(
                f"{class_name}/{load_name} "
                f"chỉ có {len(files)} file, "
                f"cần ít nhất {n_needed}."
            )

        selected = files[
            :n_needed
        ]

        train_files = (
            selected[
                :n_train
            ]
        )

        val_files = (
            selected[
                n_train:
                n_train + n_val
            ]
        )

        test_files = (
            selected[
                n_train + n_val:
            ]
        )

        class_split[
            "train"
        ].extend(
            [
                (
                    path,
                    load_name
                )
                for path in train_files
            ]
        )

        class_split[
            "validation"
        ].extend(
            [
                (
                    path,
                    load_name
                )
                for path in val_files
            ]
        )

        class_split[
            "test"
        ].extend(
            [
                (
                    path,
                    load_name
                )
                for path in test_files
            ]
        )

        print(
            f"{load_name:4s} | "
            f"Train={len(train_files):2d} | "
            f"Val={len(val_files):2d} | "
            f"Test={len(test_files):2d} | "
            f"Tổng={n_needed:2d}"
        )

    fault_splits[
        class_name
    ] = class_split

    print(
        "Tổng lớp:",
        "Train=",
        len(
            class_split[
                "train"
            ]
        ),
        "| Val=",
        len(
            class_split[
                "validation"
            ]
        ),
        "| Test=",
        len(
            class_split[
                "test"
            ]
        ),
        "| Tổng=",
        sum(
            len(
                class_split[
                    split_name
                ]
            )
            for split_name in [
                "train",
                "validation",
                "test"
            ]
        )
    )


In [ ]:

# Ô 7 - Kiểm tra không có cùng file nguồn ở nhiều split

split_source_paths = {
    "train": set(),
    "validation": set(),
    "test": set(),
}


def add_and_check(
    source_path,
    split_name
):

    source_resolved = str(
        source_path.resolve()
    )

    for other_split, source_set in (
        split_source_paths.items()
    ):

        if (
            other_split != split_name
            and source_resolved in source_set
        ):

            raise RuntimeError(
                f"Data leakage: "
                f"{source_path} nằm ở "
                f"{other_split} và {split_name}"
            )

    split_source_paths[
        split_name
    ].add(
        source_resolved
    )


for path in normal_train_files:
    add_and_check(
        path,
        "train"
    )

for path in normal_val_files:
    add_and_check(
        path,
        "validation"
    )

for path in normal_test_files:
    add_and_check(
        path,
        "test"
    )


for class_name, split_dict in (
    fault_splits.items()
):

    for split_name, items in (
        split_dict.items()
    ):

        for path, load_name in items:

            add_and_check(
                path,
                split_name
            )


print(
    "Không phát hiện file nguồn trùng "
    "giữa Train / Validation / Test."
)


In [ ]:

# Ô 8 - Copy file và tạo manifest

rows = []

# ============================================================
# NORMAL
# ============================================================

normal_split_map = {
    "train":
        normal_train_files,

    "validation":
        normal_val_files,

    "test":
        normal_test_files,
}

for split_name, files in (
    normal_split_map.items()
):

    for source_path in files:

        destination_dir = (
            OUTPUT_ROOT
            / split_name
            / "normal"
        )

        if COPY_FILES:

            destination_path = (
                copy_one_file(
                    source_path,
                    destination_dir
                )
            )

        else:

            destination_path = (
                destination_dir
                / source_path.name
            )

        rows.append(
            {
                "class":
                    "normal",

                "load":
                    "normal",

                "split":
                    split_name,

                "source_file":
                    str(
                        source_path
                    ),

                "destination_file":
                    str(
                        destination_path
                    ),
            }
        )


# ============================================================
# FAULTS
# ============================================================

for class_name, split_dict in (
    fault_splits.items()
):

    for split_name, items in (
        split_dict.items()
    ):

        for source_path, load_name in items:

            destination_dir = (
                OUTPUT_ROOT
                / split_name
                / class_name
                / load_name
            )

            if COPY_FILES:

                destination_path = (
                    copy_one_file(
                        source_path,
                        destination_dir
                    )
                )

            else:

                destination_path = (
                    destination_dir
                    / source_path.name
                )

            rows.append(
                {
                    "class":
                        class_name,

                    "load":
                        load_name,

                    "split":
                        split_name,

                    "source_file":
                        str(
                            source_path
                        ),

                    "destination_file":
                        str(
                            destination_path
                        ),
                }
            )


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

manifest_df = pd.DataFrame(
    rows
)

manifest_path = (
    OUTPUT_ROOT
    / "split_manifest.csv"
)

manifest_df.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Đã lưu manifest:",
    manifest_path
)


In [ ]:

# Ô 9 - Kiểm tra cân bằng cuối cùng

count_table = (
    manifest_df
    .groupby(
        [
            "split",
            "class"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

display(
    count_table
)

total_by_class = (
    manifest_df
    .groupby(
        "class"
    )
    .size()
)

print(
    "\nTổng file mỗi lớp:"
)

print(
    total_by_class
)

# Kiểm tra bắt buộc
expected_split_counts = {
    "train": 34,
    "validation": 7,
    "test": 8,
}

for split_name, expected in (
    expected_split_counts.items()
):

    for class_name in [
        "normal",
        "inner_race",
        "outer_race",
        "ball_fault",
    ]:

        actual = int(
            count_table.loc[
                split_name,
                class_name
            ]
        )

        if actual != expected:

            raise RuntimeError(
                f"{split_name}/{class_name}: "
                f"{actual}, cần {expected}."
            )


for class_name in [
    "normal",
    "inner_race",
    "outer_race",
    "ball_fault",
]:

    if int(
        total_by_class[
            class_name
        ]
    ) != 49:

        raise RuntimeError(
            f"{class_name} không đủ 49 file."
        )


print(
    "\nCÂN BẰNG ĐÚNG:"
)

print(
    "Train      = 34 file/lớp"
)

print(
    "Validation = 7 file/lớp"
)

print(
    "Test       = 8 file/lớp"
)

print(
    "Tổng       = 49 file/lớp"
)



## Kết quả

Sau khi chạy, cấu trúc:

```text
balanced_raw_split_49_each_class/
├── train/
│   ├── normal/                  # 34 file
│   ├── inner_race/              # 34 file
│   ├── outer_race/              # 34 file
│   └── ball_fault/              # 34 file
├── validation/
│   ├── normal/                  # 7 file
│   ├── inner_race/              # 7 file
│   ├── outer_race/              # 7 file
│   └── ball_fault/              # 7 file
└── test/
    ├── normal/                  # 8 file
    ├── inner_race/              # 8 file
    ├── outer_race/              # 8 file
    └── ball_fault/              # 8 file
```

Mỗi lớp có đúng:

\[
34+7+8=49\text{ file}
\]

Không oversample Normal và không có file nguồn trùng giữa Train/Validation/Test.
